# 📈 Stock Market Data Analyzer: Quantitative Research & Backtesting
**Author:** Student Financial Data Analyst / Python Developer  
**Objective:** End-to-end financial time-series ingestion, technical indicator computation, risk modeling, and algorithmic backtesting.

### Step 1: Environment Setup & Library Imports

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_fetcher import StockDataFetcher
from src.indicators import TechnicalIndicators
from src.risk_metrics import RiskMetricsEngine
from src.backtester import StrategyBacktester

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

### Step 2: Fetching Market Data (AAPL / NVDA / TSLA / SPY)

In [ ]:
fetcher = StockDataFetcher(data_dir='../data')
df_aapl = fetcher.fetch_data('AAPL', period='2y')
df_aapl.tail()

### Step 3: Computing Technical Indicators (SMA, RSI, MACD, Bollinger Bands)

In [ ]:
df_analyzed = TechnicalIndicators.add_all_indicators(df_aapl)
df_analyzed[['Close', 'SMA_20', 'SMA_50', 'SMA_200', 'RSI_14', 'MACD', 'Overall_Trend']].tail()

### Step 4: Quantitative Risk & Return Profiling

In [ ]:
risk_engine = RiskMetricsEngine(risk_free_rate=0.04)
metrics = risk_engine.calculate_all_metrics(df_analyzed)
for k, v in metrics.items():
    print(f"{k.replace('_', ' ').title()}: {v}")

### Step 5: Strategy Backtesting (Dual MA Crossover vs Buy & Hold)

In [ ]:
backtester = StrategyBacktester(initial_capital=10000.0)
bt_result = backtester.backtest_ma_crossover(df_analyzed, fast_period=20, slow_period=50)

print(f"Initial Capital: ${bt_result['initial_capital']}")
print(f"Final Strategy Equity: ${bt_result['final_strategy_equity']:,.2f} ({bt_result['strategy_total_return_pct']}%)")
print(f"Final Benchmark Equity: ${bt_result['final_benchmark_equity']:,.2f} ({bt_result['benchmark_total_return_pct']}%)")
print(f"Alpha Outperformance: {bt_result['alpha_pct']}%")
print(f"Win Rate: {bt_result['win_rate_pct']}%")

### Step 6: Equity Growth Curve Visualization

In [ ]:
eq_df = bt_result['equity_df']
plt.figure(figsize=(12, 6))
plt.plot(eq_df.index, eq_df['Strategy_Equity'], label=f"Strategy: MA Crossover ({bt_result['strategy_total_return_pct']}%)", color='green', linewidth=2)
plt.plot(eq_df.index, eq_df['Benchmark_Equity'], label=f"Buy & Hold Benchmark ({bt_result['benchmark_total_return_pct']}%)", color='blue', linestyle='--')
plt.title('AAPL - Strategy Equity Curve vs Benchmark ($10,000 Portfolio)', fontsize=14, fontweight='bold')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Date')
plt.legend()
plt.show()